# Preprocessing Teks Ulasan E-commerce Berbahasa Indonesia

**Nama:** ARIL AL IQBAL 
**NIM:** 24146034 
**Mata kuliah:** Natural Language Processing (NLP)

Notebook ini memandu langkah demi langkah preprocessing teks ulasan e-commerce
berbahasa Indonesia, mulai dari memuat dataset sampai membandingkan statistik
teks sebelum dan sesudah preprocessing.

**Dataset:** `dipawidia/ecommerce-product-reviews-sentiment`  
**Label:** `0` = negatif, `1` = positif  
**Split yang dipakai:** `train`

## Cara Menjalankan (Google Colab / Lokal)

**Di Google Colab:**
1. Upload file `.ipynb` ini, lalu pilih **Runtime → Restart session** agar sesi bersih.
2. Jalankan sel **Persiapan Library** di bawah (perintah instal sudah disiapkan).
3. Pilih **Runtime → Run all**. Jalankan dari atas ke bawah; jangan membuka sel demo
   sebelum sel definisi fungsinya.
4. Jika Colab meminta restart setelah instal, pilih **Runtime → Restart session**,
   lalu **Run all** lagi.

**Di komputer lokal:** buka dengan Jupyter/VS Code, pilih kernel yang benar, lalu
jalankan semua sel dari atas ke bawah.

> Notebook ini mengunduh dataset dari internet dan menjalankan stemming Sastrawi.
> Proses stemming pada seluruh data bisa memakan beberapa menit.

## Persiapan Library

Sel ini memasang library yang dibutuhkan ke environment kernel yang sedang aktif.
`%pip` dipakai (bukan `!pip`) supaya paket pasti terpasang di Python yang dipakai
notebook, termasuk di Google Colab.

In [ ]:
# Memasang library yang belum tersedia (aman dijalankan berulang)
%pip install -q datasets Sastrawi

# Mengecek bahwa library bisa diimpor
import datasets
import Sastrawi
import pandas
import matplotlib

print("datasets   :", datasets.__version__)
print("pandas     :", pandas.__version__)
print("matplotlib :", matplotlib.__version__)
print("Sastrawi   : siap")

> Jika muncul pesan **"You must restart the runtime"**, lakukan
> **Runtime → Restart session**, lalu jalankan lagi dari sel ini. Ini normal
> ketika Colab memperbarui library yang sudah terpasang.

## 1. Memuat Dataset

Tujuan langkah ini adalah mengunduh dataset dari HuggingFace dan mengubahnya
menjadi pandas DataFrame. DataFrame dipakai karena bentuknya seperti tabel
(baris = ulasan, kolom = atribut), sehingga lebih mudah diperiksa dan diolah.

In [ ]:
# Mengimpor library yang dibutuhkan
import re
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

# Mengunduh dataset HuggingFace dan langsung mengambil split "train"
dataset = load_dataset("dipawidia/ecommerce-product-reviews-sentiment", split="train")

# Mengubah dataset menjadi pandas DataFrame
df = dataset.to_pandas()

# Menampilkan 5 baris pertama untuk memastikan data berhasil dimuat
df.head()

## 2. Informasi dan Distribusi Dataset

Sebelum mengolah teks, kita perlu tahu berapa banyak data dan bagaimana sebaran
label sentimennya. Informasi ini penting agar kita sadar apakah data seimbang;
jika sangat tidak seimbang, akurasi model bisa terlihat tinggi padahal model
hanya menebak kelas mayoritas. Grafik batang membantu melihat perbandingannya
secara cepat.

In [ ]:
# Menampilkan jumlah data dan nama kolom
print("Jumlah data :", len(df))
print("Nama kolom  :", df.columns.tolist())
print()

# Menghitung jumlah dan persentase tiap label sentimen
jumlah = df["sentimen"].value_counts().sort_index()
persen = df["sentimen"].value_counts(normalize=True).sort_index() * 100

# Merangkum distribusi label dalam tabel
ringkasan = pd.DataFrame({
    "Kode": jumlah.index,
    "Sentimen": ["Negatif" if kode == 0 else "Positif" for kode in jumlah.index],
    "Jumlah": jumlah.values,
    "Persentase (%)": persen.round(2).values
})

ringkasan

In [ ]:
# Mengambil jumlah tiap label, diurutkan dari 0 ke 1
jumlah = df["sentimen"].value_counts().sort_index()

# Membuat grafik batang
ax = jumlah.plot(kind="bar", color=["#e74c3c", "#2ecc71"])
ax.set_title("Distribusi Sentimen Ulasan")
ax.set_xlabel("Sentimen (0 = negatif, 1 = positif)")
ax.set_ylabel("Jumlah ulasan")
ax.set_xticklabels(["Negatif (0)", "Positif (1)"], rotation=0)

# Menulis angka jumlah di atas setiap batang
for i, nilai in enumerate(jumlah.values):
    ax.text(i, nilai + 80, str(nilai), ha="center")

plt.tight_layout()
plt.show()

## 3. Preprocessing Teks

Preprocessing adalah proses merapikan teks agar lebih seragam dan mudah
dianalisis. Kita memecahnya menjadi fungsi-fungsi kecil agar setiap tahap mudah
dipahami, diuji, dan diperbaiki. Urutannya penting: case folding dan cleaning
dilakukan sebelum tokenisasi, lalu stopword removal dan stemming pada level token.

> **Penting:** setiap subbab punya sel **definisi fungsi** lalu sel **demo**.
> Jalankan sel definisi terlebih dahulu. Jika memakai **Run all**, urutan ini
> sudah otomatis benar.

### 3.1 Case Folding

Case folding mengubah semua huruf menjadi huruf kecil. Tanpa langkah ini, kata
"Bagus" dan "bagus" akan dianggap sebagai dua kata yang berbeda, sehingga kosa
kata menjadi lebih besar dari yang seharusnya.

In [ ]:
def case_folding(teks):
    # Mengubah seluruh huruf di teks menjadi huruf kecil
    return teks.lower()

In [ ]:
# Mengambil satu contoh ulasan
contoh = df.loc[0, "review"]

print("Sebelum :", contoh)
print("Sesudah :", case_folding(contoh))

### 3.2 Cleaning

Cleaning membuang bagian teks yang tidak membantu analisis, seperti URL, angka,
tanda baca, dan karakter khusus. Tujuannya supaya yang tersisa hanya huruf dan
spasi, sehingga token yang dihasilkan lebih bersih dan konsisten.

In [ ]:
def cleaning(teks):
    # Menghapus URL, angka, tanda baca, dan karakter khusus
    teks = re.sub(r"http\S+|www\.\S+", " ", teks)  # hapus URL
    teks = re.sub(r"\d+", " ", teks)               # hapus angka
    teks = re.sub(r"[^a-z\s]", " ", teks)          # sisakan huruf a-z dan spasi
    teks = re.sub(r"\s+", " ", teks)               # rapikan spasi berlebih
    return teks.strip()

In [ ]:
contoh = case_folding(df.loc[0, "review"])

print("Sebelum :", contoh)
print("Sesudah :", cleaning(contoh))

### 3.3 Tokenisasi

Tokenisasi memecah teks menjadi daftar kata (token). Tahap ini diperlukan karena
stopword removal dan stemming bekerja per kata, bukan pada kalimat utuh. Di sini
kita memakai pemisahan sederhana berdasarkan spasi.

In [ ]:
def tokenisasi(teks):
    # Memecah teks menjadi daftar kata berdasarkan spasi
    return teks.split()

In [ ]:
contoh = cleaning(case_folding(df.loc[0, "review"]))
tokens = tokenisasi(contoh)

print("Teks  :", contoh)
print("Token :", tokens)
print("Jumlah token:", len(tokens))

### 3.4 Stopword Removal dengan Sastrawi

Stopword adalah kata umum yang sering muncul tetapi tidak banyak memberi makna,
misalnya "dan", "di", atau "yang". Sastrawi menyediakan daftar stopword bahasa
Indonesia, sehingga kita bisa membuang kata-kata tersebut dari daftar token.

In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Mengambil daftar stopword bahasa Indonesia dari Sastrawi
stopword_sastrawi = set(StopWordRemoverFactory().get_stop_words())

def hapus_stopword(tokens):
    # Membuang token yang termasuk stopword Sastrawi
    return [kata for kata in tokens if kata not in stopword_sastrawi]

In [ ]:
tokens = tokenisasi(cleaning(case_folding(df.loc[0, "review"])))

print("Sebelum :", tokens)
print("Sesudah :", hapus_stopword(tokens))

> **Catatan:** daftar stopword Sastrawi juga memuat kata negasi seperti "tidak".
> Untuk analisis sentimen, kata negasi sebaiknya dipertahankan karena bisa
> membalik makna kalimat. Di tugas ini kita tetap memakai daftar standar Sastrawi
> sesuai instruksi.

### 3.5 Stemming dengan Sastrawi dan Cache

Stemming mengubah kata berimbuhan menjadi kata dasarnya, misalnya "pengiriman"
menjadi "kirim". Proses ini berat jika setiap kemunculan kata dihitung ulang,
jadi kita menyimpan hasilnya di sebuah dictionary cache per kata unik. Dengan
begitu, kata yang sama hanya di-stem satu kali.

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Membuat objek stemmer Sastrawi
stemmer = StemmerFactory().create_stemmer()

# Cache: tempat menyimpan hasil stemming setiap kata unik
cache_stemming = {}

def stemming(tokens):
    # Mengubah token menjadi kata dasar menggunakan cache
    hasil = []

    for kata in tokens:
        if kata not in cache_stemming:                  # kata belum pernah di-stem
            cache_stemming[kata] = stemmer.stem(kata)   # hitung sekali saja
        hasil.append(cache_stemming[kata])              # ambil hasil dari cache

    return hasil

In [ ]:
tokens = hapus_stopword(tokenisasi(cleaning(case_folding(df.loc[0, "review"]))))

print("Sebelum :", tokens)
print("Sesudah :", stemming(tokens))
print("Jumlah kata di cache:", len(cache_stemming))

### 3.6 Menggabungkan Semua Tahap

Sekarang kita gabungkan kelima fungsi di atas menjadi satu fungsi `preprocess`.
Dengan satu fungsi ini, seluruh tahap bisa dijalankan berurutan pada setiap
ulasan tanpa perlu menulis ulang kodenya.

In [ ]:
def preprocess(teks):
    # Menjalankan seluruh tahap preprocessing secara berurutan
    if not isinstance(teks, str):      # jaga-jaga jika ada data kosong
        return []

    teks = case_folding(teks)
    teks = cleaning(teks)
    tokens = tokenisasi(teks)
    tokens = hapus_stopword(tokens)
    tokens = stemming(tokens)

    return tokens

In [ ]:
contoh = df.loc[0, "review"]

print("Sebelum :", contoh)
print("Sesudah :", " ".join(preprocess(contoh)))

In [28]:
# Menjalankan preprocessing pada seluruh ulasan
semua_tokens = []

for i, teks in enumerate(df["review"], start=1):
    semua_tokens.append(preprocess(teks))

    # Menampilkan progres setiap 1000 ulasan
    if i % 1000 == 0:
        print(f"{i} ulasan selesai diproses...")

# Menyimpan hasil ke DataFrame
df["tokens_bersih"] = pd.Series(semua_tokens, index=df.index, dtype="object")
df["teks_bersih"] = df["tokens_bersih"].apply(lambda token: " ".join(token))

print("Selesai! Total:", len(df), "ulasan")
print("Jumlah kata unik di cache stemming:", len(cache_stemming))
df[["review", "teks_bersih"]].head()

11000 ulasan selesai diproses...
Selesai! Total: 11606 ulasan
Jumlah kata unik di cache stemming: 12436


,review,teks_bersih
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,suara mic puas pesan yg nyala sy pasti lg klik...
1,Good. Mudah-mudahan awet. Makasih,good mudah mudah awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek perhati
3,Kualitas produk baik. Harga terjangkau dan pen...,kualitas produk baik harga jangkau kirim cepet...
4,"Overall bagus, tombol berfungsi semuanya, tapi...",overall bagus tombol fungsi semua packaging ku...


## 4. Contoh Hasil Sebelum dan Sesudah Preprocessing

Bagian ini menampilkan 10 ulasan acak sebelum dan sesudah preprocessing dalam
satu tabel. Tujuannya agar kita bisa memeriksa langsung apakah pembersihan sudah
sesuai harapan dan apakah ada informasi penting yang ikut terbuang.

In [29]:
# Mengatur lebar kolom agar teks mudah dibaca
pd.set_option("display.max_colwidth", 100)

# Mengambil 10 ulasan acak agar contohnya lebih beragam
contoh_tabel = df.sample(10, random_state=42)[["sentimen", "review", "teks_bersih"]].copy()

# Mengubah kode sentimen menjadi nama yang mudah dibaca
contoh_tabel["sentimen"] = contoh_tabel["sentimen"].map({0: "Negatif", 1: "Positif"})

# Menambahkan nomor urut dan mengganti nama kolom
contoh_tabel.insert(0, "No", range(1, 11))
contoh_tabel = contoh_tabel.rename(columns={
    "sentimen": "Sentimen",
    "review": "Sebelum",
    "teks_bersih": "Sesudah"
})

contoh_tabel

,No,Sentimen,Sebelum,Sesudah
8077,1,Negatif,Barangnya original 👍. Bisa muat banyak. Gampang dipakai. Penjualnya kurang ramah,barang original muat banyak gampang pakai jual kurang ramah
10959,2,Positif,Barang sudah sampai pengirimannya cepat,barang kirim cepat
5786,3,Positif,"realpict, packing oke",realpict packing oke
2503,4,Negatif,Cuman bisa komen 👍👍👍,cuman komen
10113,5,Positif,"mantap banget, pengiriman cepat pesan sekarang besok sudah ditangan.Desain elegan 😍. Resolusi ga...",mantap banget kirim cepat pesan sekarang besok tang desain elegan resolusi gambar tajam
4299,6,Negatif,"awalnya dikrm hanya unit saja, setelah komplain baru dikrm dan info bonus bundling harusnya diup...",awal dikrm unit komplain baru dikrm info bonus bundling harus diupdate kalo tumbler dah habis ja...
10333,7,Negatif,"menurut infonya free Mola TV tp ternyata perlu bayar, setelah diskusi panjang akhirnya diketahui...",info free mola tv tp nyata perlu bayar diskusi panjang akhir tahu kalau os nya diupdate beli har...
4877,8,Negatif,Pesanan sesuai cuma syang nya gak ada saringanya buat mengkukus lain kali lebih teliti lgi ya,pesan sesuai cuma syang nya gak saringanya buat kukus kali lebih teliti lgi
8239,9,Negatif,remote control tidak peka,remote control peka
7078,10,Positif,Sudah sampai mesin cuci nya terima kasih cepat ya 5 hari sampai...kerennn,mesin cuci nya terima kasih cepat hari kerennn


## 5. Statistik Sebelum vs Sesudah Preprocessing

Kita hitung rata-rata jumlah kata dan jumlah kosa kata unik untuk melihat dampak
preprocessing. Kosa kata unik adalah banyaknya kata berbeda di seluruh dataset.
Biasanya rata-rata jumlah kata sedikit berkurang, sedangkan kosa kata unik turun
lebih banyak karena variasi bentuk kata dan noise sudah dihilangkan.

In [30]:
# SEBELUM: teks asli dipecah menjadi kata (di-lowercase agar adil saat dihitung)
kata_sebelum = df["review"].str.lower().str.split()
rata_kata_sebelum = kata_sebelum.apply(len).mean()
vocab_sebelum = set(kata for tokens in kata_sebelum for kata in tokens)

# SESUDAH: token hasil preprocessing
rata_kata_sesudah = df["tokens_bersih"].apply(len).mean()
vocab_sesudah = set(kata for tokens in df["tokens_bersih"] for kata in tokens)

# Merangkum perbandingan dalam tabel
statistik = pd.DataFrame({
    "Tahap": ["Sebelum preprocessing", "Sesudah preprocessing"],
    "Rata-rata jumlah kata": [round(rata_kata_sebelum, 2), round(rata_kata_sesudah, 2)],
    "Jumlah kosa kata unik": [len(vocab_sebelum), len(vocab_sesudah)]
})

statistik

,Tahap,Rata-rata jumlah kata,Jumlah kosa kata unik
0,Sebelum preprocessing,15.76,23371
1,Sesudah preprocessing,13.06,10161


In [31]:
# Menghitung besarnya penurunan kosa kata unik
selisih = len(vocab_sebelum) - len(vocab_sesudah)
persen_turun = selisih / len(vocab_sebelum) * 100

print(f"Kosa kata unik berkurang {selisih} kata ({persen_turun:.2f}%).")

Kosa kata unik berkurang 13210 kata (56.52%).


> **Catatan:** pada teks asli, tanda baca masih menempel pada kata, sehingga
> jumlah kosa kata "sebelum" terlihat lebih besar. Perbandingan ini tetap berguna
> untuk menunjukkan efek pembersihan dan penyeragaman bentuk kata.

## Kesimpulan

Preprocessing berhasil membuat teks lebih seragam: huruf disamakan, noise
dibuang, kata umum dihilangkan, dan kata berimbuhan dikembalikan ke bentuk dasar.
Hasilnya, rata-rata jumlah kata dan kosa kata unik menurun, sehingga data lebih
ringkas dan siap dipakai untuk tahap berikutnya seperti TF-IDF dan klasifikasi
sentimen.

## 3 Pertanyaan Latihan

1. Mengapa case folding sebaiknya dilakukan sebelum cleaning dan tokenisasi?
   Apa yang bisa salah jika urutannya dibalik?
2. Stopword removal Sastrawi ikut membuang kata negasi seperti "tidak" dan
   "bukan". Jelaskan dampaknya untuk analisis sentimen, lalu usulkan solusi.
3. Lihat tabel statistik. Mengapa jumlah kosa kata unik turun jauh lebih besar
   dibandingkan rata-rata jumlah kata per ulasan? Kaitkan jawabanmu dengan
   peran stemming dan stopword removal.